# W13-D1 概念实验：People Analytics 五指标 × ID 依赖度谱系

配套阅读材料：`第13周-Day1-PeopleAnalytics-客流为什么是Tracking.md`

md 讲的是论断（**计数是事件，不是状态**），本 notebook 用可执行实验定量验证：

**实验设计（控制变量）**
- 模拟一个商场入口的 2D 世界：入口线 x=0.5，右侧 ROI = 店内
- 行人类型：快速通过者 ×10、徘徊者 ×1、来回折返者 ×1、静止保安 ×1、广告假人 ×1
- "有 Tracking" = oracle ID（真实身份直接可用，代表 Tracking 上界）
- "无 ID" = 检测视图：每帧一袋无标签的点（模拟完美检测器 + 5% 漏检 + σ=0.01 噪声）
- **关键控制**：检测器质量两边完全相同——唯一变量是 ID 的有无

**待验证的谱系假说**（md §9）
| 指标 | 类型 | 无 ID 后果 |
|---|---|---|
| 密度 | 累积型（状态） | 误差 ≈ 检测噪声，优雅降级 |
| 热区 | 累积型（状态） | 同上 |
| 客流（过线计数） | 事件型 | 结构性偏差（不是噪声） |
| 停留时间 | 持续型（事件） | 退化为 ROI 占用时长，连续人流下爆炸 |
| 轨迹 | ID 即指标 | 信息论不可恢复 |

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体就绪:", font_name)

## 实验 1：模拟商场入口世界

构造 5 类真实轨迹（truth）。注意两类"陷阱"：
- **保安**（左侧静止不动）→ 贡献 person-frames 但不过线
- **广告假人**（ROI 内静止、出生即在店内）→ 贡献 person-frames 但**从未"进入"**（无入场事件）

In [ ]:
T = 200          # 帧数
LINE_X = 0.5     # 入口计数线（竖直）
rng = np.random.default_rng(42)

def walk_reflect(pos, lo, hi, step=0.02):
    """带边界反射的随机游走（模拟店内徘徊）。"""
    p = pos + rng.normal(0, step)
    if p < lo or p > hi:
        p = np.clip(pos - (p - pos), lo, hi)
    return float(np.clip(p, lo, hi))

persons = []  # dict: pid, kind, birth, death, traj[T,2]（生命期外为 nan）

def add_person(pid, kind, birth, death, traj):
    full = np.full((T, 2), np.nan)
    full[birth:death] = traj
    persons.append(dict(pid=pid, kind=kind, birth=birth, death=death, traj=full))

# 1) 快速通过者 ×10：左→右直线，错峰出生
for i in range(10):
    dur = 40
    birth = i * 8
    x = np.linspace(0.02, 0.98, dur) + rng.normal(0, 0.003, dur)
    y = np.full(dur, rng.uniform(0.1, 0.9)) + rng.normal(0, 0.003, dur)
    add_person(100 + i, "passer", birth, birth + dur, np.c_[x, y])

# 2) 徘徊者：过线后在店内随机游走（长停留）
dur = 120
traj = np.full((dur, 2), np.nan)
x, y = 0.1, 0.5
for t in range(dur):
    x = 0.02 + 0.92 * (t / 10) if t < 10 else walk_reflect(x, 0.55, 0.95)
    y = 0.5 if t < 10 else walk_reflect(y, 0.2, 0.8)
    traj[t] = x, y
add_person(200, "lingerer", 20, 20 + dur, traj)

# 3) 来回折返者：x 围绕入口线正弦摆动（多次过线）
dur = 190
t = np.arange(dur)
add_person(201, "shuttler", 10, 10 + dur,
           np.c_[0.5 + 0.25 * np.sin(2 * np.pi * (t + 15) / 60), np.full(dur, 0.5)])

# 4) 静止保安：左侧固定位置（有生命期，全程在场）
add_person(300, "guard", 0, T, np.c_[np.full(T, 0.30) + rng.normal(0, 0.002, T),
                                     np.full(T, 0.75) + rng.normal(0, 0.002, T)])

# 5) 广告假人：ROI 内固定位置，出生即在店内（无入场事件）
add_person(301, "mannequin", 0, T, np.c_[np.full(T, 0.82) + rng.normal(0, 0.002, T),
                                         np.full(T, 0.25) + rng.normal(0, 0.002, T)])

print(f"共 {len(persons)} 人：", {p['kind']: sum(1 for q in persons if q['kind'] == p['kind']) for p in persons})

## 生成"检测视图"：完美检测器 + 5% 漏检 + 噪声

检测器质量对两条路线完全一致（同一随机种子）。漏检/噪声代表现实误差；
关键是观察：**累积型指标的误差是线性的（噪声级），事件型指标的误差是结构性的（偏差级）**。

In [ ]:
DET_RATE, DET_SIGMA = 0.95, 0.01

detections = []  # (frame, x, y) — 无 ID 点袋
for p in persons:
    for f in range(p["birth"], p["death"]):
        if rng.random() < DET_RATE:
            x, y = p["traj"][f] + rng.normal(0, DET_SIGMA, 2)
            detections.append((f, x, y))
det_arr = np.array(detections)
frames_det = det_arr[:, 0].astype(int)

print(f"真值总采样（人×帧）: {sum(p['death'] - p['birth'] for p in persons)}")
print(f"检测视图点数（~{DET_RATE:.0%} 检出率）: {len(det_arr)}")

## 实验 2a：密度与热区（假说：累积型指标可无 ID 近似）

In [ ]:
def in_roi(xy):  # ROI = 店内（x > 0.5）
    return xy[:, 0] > LINE_X

# --- 密度：逐帧 ROI 内人数 ---
true_density = np.zeros(T)
for p in persons:
    true_density += np.nan_to_num(in_roi(p["traj"]), nan=0.0)
det_density = np.zeros(T)
np.add.at(det_density, frames_det[in_roi(det_arr[:, 1:])], 1)

density_err = np.abs(det_density - true_density).mean() / max(true_density.mean(), 1)
print(f"真实平均店内密度: {true_density.mean():.2f} 人/帧")
print(f"无ID 密度相对误差: {density_err:.1%}（≈漏检率，噪声级）")

# --- 热区：20×20 网格累积 ---
G = 20
edges = np.linspace(0, 1, G + 1)
truth_pts = np.vstack([p["traj"][~np.isnan(p["traj"][:, 0])] for p in persons])
H_true, _, _ = np.histogram2d(truth_pts[:, 0], truth_pts[:, 1], bins=[edges, edges])
H_det, _, _ = np.histogram2d(det_arr[:, 1], det_arr[:, 2], bins=[edges, edges])  # 列=(frame,x,y)，取 x,y
cos = float(np.dot(H_true.ravel(), H_det.ravel()) /
            (np.linalg.norm(H_true.ravel()) * np.linalg.norm(H_det.ravel())))
heatmap_err = 1 - cos
print(f"无ID 热区余弦相似度: {cos:.4f} → 相对误差 {heatmap_err:.1%}（噪声级）")

## 实验 2b：客流（过线计数）—— 无 ID 的结构性偏差

**真值**（需要 ID 才能定义）：入场事件 = 从线外进入 ROI 的人次。
**无 ID 启发式**：`总 person-frames ÷ 假设的平均在场时长`。
给它最有利的假设（恰好等于真实入场者均值）——看误差还剩多少。
（ERP 语言：没有单据号的库存，只能靠"估算平均在库天数"倒轧——盘盈盘亏。）

In [ ]:
# --- 真值：入场事件与过线次数（oracle ID）---
# 入场事件 = 逐帧 outside→inside 转移次数（带方向的穿越事件，需要 ID 才能定义）
entries, crossings = 0, 0
dwell_true = {}
for p in persons:
    xs = p["traj"][p["birth"]:p["death"], 0]      # 仅生命期内
    inside = xs > LINE_X
    side = np.sign(xs - LINE_X)
    crossings += int((np.diff(side) != 0).sum())
    entries += int(((~inside[:-1]) & inside[1:]).sum())
    if inside.any():
        dwell_true[p["pid"]] = int(inside.sum())

# 无 ID 启发式（最有利假设：恰好猜中真实平均在场时长）
P_frames = len(det_arr)
mean_presence = np.mean([p["death"] - p["birth"] for p in persons
                         if np.nanmax(p["traj"][:, 0]) > LINE_X and np.nanmin(p["traj"][:, 0]) <= LINE_X])
naive_est = P_frames / mean_presence

count_err_none = abs(naive_est - entries) / entries
static_pf = int(sum(p["death"] - p["birth"] for p in persons if p["kind"] in ("guard", "mannequin")))
print(f"真实入场事件: {entries} 人次（过线总次数 {crossings}，含折返往返）")
print(f"检测视图总 person-frames: {P_frames}")
print(f"无ID 启发式估计: {naive_est:.1f} 人 → 相对误差 +{count_err_none:.0%}")
print(f"偏差来源：保安+假人贡献 {static_pf} person-frames（占 {static_pf/P_frames:.0%}）却 0 次入场——静态误报被逐帧线性放大")
print("有ID（oracle）: 误差 0%（事件按身份记账）")

## 实验 2c：停留时间 —— MallSenseAI 退化形态的失真边界

无 ID 时"人的停留"退化为"**ROI 占用时长**"（`ObstructionRuleEngine` 的 first-seen 语义）。
两个场景对照：
- **场景 A（单事件）**：一次只有一个人的占用 → 退化形态 = 真值 ✓
- **场景 B（连续人流）**：ROI 从不清空 → 占用时长 ≈ 全天，完全失真 ✗

In [ ]:
# --- 场景 B：主模拟即连续人流 ---
det_in_roi = np.zeros(T, dtype=bool)
det_in_roi[np.unique(frames_det[in_roi(det_arr[:, 1:])])] = True
occupied_span = np.flatnonzero(det_in_roi)[[0, -1]]
deg_dwell_B = occupied_span[1] - occupied_span[0]
true_dwell_mean = np.mean(list(dwell_true.values()))
dwell_err_B = abs(deg_dwell_B - true_dwell_mean) / true_dwell_mean
print(f"场景B（连续人流）：ROI 占用时长 {deg_dwell_B} 帧 vs 真实人均停留 {true_dwell_mean:.0f} 帧 → +{dwell_err_B:.0%}")

# --- 场景 A：单事件 mini-sim（一个人独占 ROI 一段时间）---
TA = 140
a = np.full((TA, 2), np.nan); a[50:90] = [0.7, 0.5]   # 帧50-90 独自店内
detA = [(f, 0.7 + rng.normal(0, 0.01), 0.5) for f in range(50, 90) if rng.random() < DET_RATE]
detA_in = sorted(f for f, x, y in detA if x > LINE_X)
deg_dwell_A = detA_in[-1] - detA_in[0]
print(f"场景A（单事件）：退化占用时长 {deg_dwell_A} 帧 vs 真实停留 40 帧 → +{abs(deg_dwell_A-40)/40:.0%}")
print("→ 退化形态在『单人单事件』下等价（这正是通道障碍物场景），在『连续人流』下爆炸（这就是客流场景）")

## 实验 2d：轨迹 —— 信息论不可恢复（不是算法问题）

两条交叉路径（无噪声、无漏检），点袋在每一帧都完全相同。
"直行"和"交换"两种解释**同样自洽**——任何无 ID 算法都无法区分。
这就是为什么说轨迹的 ID 依赖度是"ID 即指标"。

In [ ]:
t60 = np.arange(60)
# 甲：左下 → 右上；乙：左上 → 右下，同速，在中心交叉
A_path = np.c_[np.linspace(0.1, 0.9, 60), np.linspace(0.1, 0.9, 60)]
B_path = np.c_[np.linspace(0.1, 0.9, 60), np.linspace(0.9, 0.1, 60)]
swap_ok = True
for f in [0, 20, 40, 59]:  # 抽查 4 帧：交换解释与直行解释的点集逐一相同
    pts_direct = {tuple(np.round(A_path[f], 6)), tuple(np.round(B_path[f], 6))}
    pts_swapped = {tuple(np.round(A_path[f], 6)), tuple(np.round(B_path[f], 6))}
    swap_ok &= (pts_direct == pts_swapped)
print(f"任一帧的无 ID 点集，'直行'与'交换'两种解释的点袋完全相同: {swap_ok}")
print("→ 轨迹在无 ID 视图下不是'误差大'，是'不存在'（100% 不可恢复）")

## 实验 3：可视化（图存同目录）

In [ ]:
OUT = "/root/learning-notebooks/第13周/"

# ---- 图1：世界 + 热图 + 轨迹歧义（2×2）----
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
ax = axes[0, 0]
for p in persons:
    xy = p["traj"][~np.isnan(p["traj"][:, 0])]
    ax.plot(xy[:, 0], xy[:, 1], lw=1.2, alpha=0.75,
            ls="--" if p["kind"] in ("guard", "mannequin") else "-",
            label=f"{p['kind']}#{p['pid']}")
ax.axvline(LINE_X, color="red", lw=2, label="入口计数线 x=0.5")
ax.axvspan(LINE_X, 1.0, color="orange", alpha=0.08)
ax.text(0.75, 0.02, "ROI=店内", color="darkorange", fontsize=11)
ax.set_title("(a) 真实轨迹（truth，含 ID）", fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8, loc="upper left")

ax = axes[0, 1]
ax.imshow(H_det.T, origin="lower", extent=[0, 1, 0, 1], cmap="hot", alpha=0.9)
ax.set_title(f"(b) 无ID 检测累积热区（余弦相似度 {cos:.3f}）", fontsize=12)

ax = axes[1, 0]
ax.imshow(H_true.T, origin="lower", extent=[0, 1, 0, 1], cmap="hot", alpha=0.9)
ax.set_title("(c) 真值占用热区（对照 b：累积型指标近似可恢复）", fontsize=12)

ax = axes[1, 1]
ax.plot(A_path[:, 0], A_path[:, 1], "o-", ms=3, color="tab:blue", label="解释1：甲直行(蓝→右上)")
ax.plot(B_path[:, 0], B_path[:, 1], "o-", ms=3, color="tab:green", label="解释1：乙直行(绿→右下)")
ax.plot([A_path[0, 0], B_path[-1, 0]], [A_path[0, 1], B_path[-1, 1]], "k--", lw=1,
        label="解释2：中心交换（同样自洽！）")
for f in [0, 20, 40]:
    ax.scatter([A_path[f, 0], B_path[f, 0]], [A_path[f, 1], B_path[f, 1]],
               s=90, facecolors="none", edgecolors="gray", zorder=5)
ax.scatter([], [], s=90, facecolors="none", edgecolors="gray", label="无ID点袋（每帧两袋相同）")
ax.set_title("(d) 轨迹歧义：无 ID 时『直行/交换』不可区分", fontsize=12)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8, loc="upper left")

fig.suptitle("W13-D1 实验：检测视图 vs 真值 —— 五指标的可恢复性", fontsize=14, y=0.995)
fig.tight_layout()
fig.savefig(OUT + "w13d1_world_heatmap_ambiguity.png", dpi=110)
plt.close(fig)
print("saved w13d1_world_heatmap_ambiguity.png")

In [ ]:
# ---- 图2：ID 依赖度谱系（核心图）----
metrics = ["密度", "热区", "客流\n(过线计数)", "停留时间\n(连续人流)", "轨迹"]
err_no_id = [density_err, heatmap_err, count_err_none, dwell_err_B, 1.0]
err_with_id = [density_err, heatmap_err, 0.0, 1 - DET_RATE, 0.0]  # oracle 上界

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 5.5))
b1 = ax.bar(x - 0.18, np.array(err_no_id) * 100, 0.34, color="#c0392b", label="无 ID（纯检测视图）")
b2 = ax.bar(x + 0.18, np.array(err_with_id) * 100, 0.34, color="#27ae60", label="有 ID（oracle tracking 上界）")
for b, v in zip(b1, err_no_id):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 3,
            "不可恢复" if v >= 1 else f"+{v*100:.0f}%", ha="center", fontsize=10, color="#c0392b", weight="bold")
for b, v in zip(b2, err_with_id):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 3, f"{v*100:.0f}%", ha="center", fontsize=9, color="#27ae60")
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel("相对误差 %（越低越可信）")
ax.set_title("People Analytics 五指标 × ID 依赖度谱系：左两格噪声级可降级，右三格结构性失败", fontsize=12)
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(err_no_id) * 100 * 1.18)
fig.tight_layout()
fig.savefig(OUT + "w13d1_id_dependency_spectrum.png", dpi=110)
plt.close(fig)
print("saved w13d1_id_dependency_spectrum.png")

In [ ]:
# ---- 图3：停留时间退化形态的失真边界 ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

ax = axes[0]
occ_A = np.zeros(TA); occ_A[50:90] = 1
ax.plot(np.arange(TA), occ_A, lw=1, color="gray", alpha=0.6)
ax.axvspan(50, 90, alpha=0.25, color="green", label="真实停留 [50,90)=40帧")
ax.axvspan(detA_in[0], detA_in[-1], alpha=0.15, color="blue",
           label=f"退化占用 [{detA_in[0]},{detA_in[-1]})={deg_dwell_A}帧")
ax.set_title("场景A 单人单事件：退化形态 ≈ 真值（通道障碍物场景）", fontsize=11)
ax.set_xlabel("帧"); ax.set_ylabel("ROI 占用"); ax.legend(fontsize=9); ax.set_ylim(-0.1, 1.4)

ax = axes[1]
ax.plot(np.arange(T), true_density, lw=1, color="gray", label="逐帧真实店内人数（从不清空）")
ax.axvspan(occupied_span[0], occupied_span[1], alpha=0.15, color="red",
           label=f"退化占用时长 {deg_dwell_B} 帧")
for i, (pid, d) in enumerate(dwell_true.items()):
    ax.scatter(np.mean([p['birth'] for p in persons if p['pid']==pid]), d + 0.3, marker="^", color="tab:blue", zorder=5)
    ax.annotate(f"#{pid}:{d}帧", (np.mean([p['birth'] for p in persons if p['pid']==pid]), d + 0.3),
                fontsize=8, ha="center", color="tab:blue")
ax.set_title(f"场景B 连续人流：退化 {deg_dwell_B} 帧 vs 真实人均 {true_dwell_mean:.0f} 帧（+{dwell_err_B:.0%}）", fontsize=11)
ax.set_xlabel("帧"); ax.set_ylabel("人数 / 停留帧数"); ax.legend(fontsize=9)

fig.suptitle("无 ID 停留时间 = ROI 占用时长：失真边界在『人流是否连续』", fontsize=13)
fig.tight_layout()
fig.savefig(OUT + "w13d1_degenerate_dwell.png", dpi=110)
plt.close(fig)
print("saved w13d1_degenerate_dwell.png")

## 结论：谱系的定量版（与 md §9 对照）

In [ ]:
rows = [
    ("密度",     "累积型（状态）", f"{density_err:.1%}",  f"{density_err:.1%}", "可无 ID 降级"),
    ("热区",     "累积型（状态）", f"{heatmap_err:.1%}",  f"{heatmap_err:.1%}", "可无 ID 降级"),
    ("客流",     "事件型",         f"+{count_err_none:.0%}", "0%",           "结构性偏差（静态误报线性放大）"),
    ("停留时间", "持续型（事件）", f"+{dwell_err_B:.0%}",  f"~{1-DET_RATE:.0%}", "退化为 ROI 占用，连续人流失真"),
    ("轨迹",     "ID 即指标",      "不可恢复",           "0%",           "信息论不可定义"),
]
header = ("指标", "类型", "无ID误差", "有ID误差", "机制")
for r in [header] + rows:
    print(f"  {r[0]:<8}{r[1]:<14}{r[2]:<14}{r[3]:<10}  {r[4]}")

print()
print("核心验证：检测器质量两边相同，唯一变量是 ID ——")
print("累积型指标（密度/热区）误差为噪声级且优雅降级；事件型指标（客流/停留）为偏差级结构性失败；")
print("轨迹不可恢复。ERP 同构：快照堆不出流水，没有单据号的库存不是账。")